# 🗂️ 04 — LDA Topic Modeling
Discover recurring themes in customer reviews.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from data.data_loader import load_sample_data
from preprocessing.cleaner import TextCleaner
from models.topic_model import LDATopicModel

df = load_sample_data()
print(f'Loaded {len(df)} reviews')

In [ ]:
# Preprocess
cleaner = TextCleaner(remove_stopwords=True, lemmatize=False)
df['tokens'] = cleaner.tokenize_series(df['text'])
token_lists  = df['tokens'].tolist()
token_lists  = [t for t in token_lists if len(t) >= 3]

print(f'Tokenized {len(token_lists)} documents')
print(f'Example tokens: {token_lists[0]}')

In [ ]:
# Find optimal number of topics
model_tmp = LDATopicModel(n_topics=5, passes=5)
scores_df  = model_tmp.find_optimal_topics(token_lists, topic_range=range(2, 9))

plt.figure(figsize=(10, 4))
plt.plot(scores_df['n_topics'], scores_df['coherence'], marker='o', color='#6366f1')
best_n = scores_df.loc[scores_df['coherence'].idxmax(), 'n_topics']
plt.axvline(x=best_n, linestyle='--', color='#f59e0b', label=f'Best: {best_n} topics')
plt.xlabel('Number of Topics')
plt.ylabel('Coherence Score (C_v)')
plt.title('Coherence Score vs Number of Topics')
plt.legend()
plt.tight_layout()
plt.savefig('../assets/coherence_plot.png', dpi=150)
plt.show()
print(f'\nBest number of topics: {best_n}')

In [ ]:
# Train final LDA model
lda = LDATopicModel(n_topics=int(best_n), passes=15)
lda.fit(token_lists)

topics = lda.get_topics(n_words=10)
print('Discovered Topics:')
print('=' * 50)
for topic in topics:
    print(f"\n{topic['label']}")
    print(f"  Keywords: {topic['representation']}")
    print(f"  Top words: {', '.join(topic['words'][:8])}")

In [ ]:
# Assign topics to reviews
df_topics = lda.assign_topics(df, text_col='text')
print('Topic distribution:')
print(df_topics['topic_label'].value_counts())

df_topics[['text', 'topic_label', 'topic_confidence']].head(10)

In [ ]:
# Compute coherence of final model
score = lda.coherence_score(token_lists)
print(f'Final Model Coherence Score: {score:.4f}')
if score > 0.55:
    print('Quality: Excellent ✅')
elif score > 0.45:
    print('Quality: Good ✅')
else:
    print('Quality: Fair — try different n_topics')

In [ ]:
# Visualize topic word weights
fig, axes = plt.subplots(1, min(3, len(topics)), figsize=(15, 4))
if len(topics) == 1:
    axes = [axes]

colors = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b', '#ef4444']

for i, (ax, topic) in enumerate(zip(axes, topics[:3])):
    ax.barh(topic['words'][:8][::-1], topic['weights'][:8][::-1],
            color=colors[i % len(colors)])
    ax.set_title(topic['label'])
    ax.set_xlabel('Weight')

plt.tight_layout()
plt.savefig('../assets/topic_words.png', dpi=150)
plt.show()